# IOAI — 2026 Summer National Constrained Generation (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
print('test_cases.json 은 노트북 첫 셀이 gdown 으로 자동 다운로드합니다. Qwen3-0.6B 생성은 GPU 권장(런타임>T4).')
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# Constrained Generation (Korlátozott Generálás)

> **HAIO 2026 — Summer National Final (NLP / generation)**

Given a `prompt`, a list of **forbidden words**, and a `max_new_tokens` budget, generate a continuation that is **fluent, diverse, and never uses a forbidden word** (any inflection — `s`/`es`/`ed`/`ing` — counts as a violation).

**Model:** `Qwen/Qwen3-0.6B-Base` (frozen). **Score per case** `= length_ratio · √distinct_2 · fluency`, `0` if any forbidden word appears; final score = mean over the 20 cases × 100.

**Submit** `submission.csv` (`id,output,fluency,tokens`). The grader recomputes the violation check and diversity from your `output`, and uses your reported `fluency` (clamped to `[0,1]`).

Baseline below: **greedy decoding that masks forbidden tokens at each step** (`select_next_token`). Improve it with beam search, look-ahead masking of word *prefixes*, or sampling for higher diversity.

In [ ]:
import os, sys, subprocess, json
# test cases (bundled locally as data/test_cases.json; on Colab fetched via gdown)
TC = 'data/test_cases.json'
if not os.path.exists(TC):
    os.makedirs('data', exist_ok=True)
    try:
        import gdown
    except ImportError:
        subprocess.run([sys.executable,'-m','pip','install','-q','gdown'], check=True); import gdown
    gdown.download(id='1vIznUAKk0uxJbRZ-jCB_aD9F63uonUz7', output=TC, quiet=True)
cases = json.load(open(TC))
print('test cases:', len(cases))

In [ ]:
import torch, torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
GEN_NAME = 'Qwen/Qwen3-0.6B-Base'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype = torch.float16 if device=='cuda' else torch.float32
gen_tokenizer = AutoTokenizer.from_pretrained(GEN_NAME)
gen_model = AutoModelForCausalLM.from_pretrained(GEN_NAME, torch_dtype=dtype).to(device).eval()
print('loaded', GEN_NAME, 'on', device)

In [ ]:
import re, math, time
SUFFIXES = r'(s|es|ed|ing)?'
def normalize_text(t): return re.sub(r'\s+',' ',t.lower()).strip()
def would_violate(text, forbidden):
    text = normalize_text(text)
    for w in forbidden:
        if re.search(r'\b'+re.escape(w.lower())+SUFFIXES+r'\b', text): return True
    return False
def distinct_2(text):
    words = re.findall(r'\w+', text.lower())
    if len(words)<2: return 0.0
    bg = list(zip(words, words[1:])); return len(set(bg))/len(bg)
@torch.no_grad()
def average_logprob(prompt, output):
    ids = gen_tokenizer.encode(prompt+output, return_tensors='pt').to(gen_model.device)
    logits = gen_model(ids).logits[:, :-1, :]
    tgt = ids[:, 1:]
    lp = F.log_softmax(logits, -1).gather(-1, tgt.unsqueeze(-1)).squeeze(-1)
    return lp.mean().item()
def score_case(prompt, forbidden, mnt, output):
    if would_violate(output, forbidden): return {'fluency':0.0,'tokens':0,'score':0.0}
    if not output.strip(): return {'fluency':0.0,'tokens':0,'score':0.0}
    tokens = len(gen_tokenizer.encode(output))
    lr = min(1.0,(tokens/mnt)**1.5); d2 = distinct_2(output)
    fl = math.exp(average_logprob(prompt, output)/5)
    return {'fluency':fl,'tokens':tokens,'score':lr*math.sqrt(d2)*fl}

In [ ]:
# --- baseline policy: greedy, masking tokens that would create a forbidden word ---
@torch.no_grad()
def forbidden_token_ids(forbidden):
    bad = set()
    for w in forbidden:
        for variant in (w, ' '+w, w.capitalize(), ' '+w.capitalize()):
            for tid in gen_tokenizer.encode(variant, add_special_tokens=False):
                bad.add(tid)
    return bad
@torch.no_grad()
def generate_constrained(prompt, forbidden, max_new_tokens):
    ids = gen_tokenizer.encode(prompt, return_tensors='pt').to(gen_model.device)
    bad = forbidden_token_ids(forbidden)
    out = gen_model(ids, use_cache=True); pkv = out.past_key_values
    logits = out.logits[0,-1,:]; gen=[]
    for _ in range(max_new_tokens):
        lg = logits.clone()
        for tid in bad:
            if tid < lg.shape[0]: lg[tid] = -1e9
        nxt = int(lg.argmax().item())
        if nxt == gen_tokenizer.eos_token_id: break
        gen.append(nxt)
        # re-check: if appending creates a forbidden word, stop early
        txt = gen_tokenizer.decode(gen)
        if would_violate(txt, forbidden): gen.pop(); break
        step = gen_model(torch.tensor([[nxt]], device=gen_model.device), past_key_values=pkv, use_cache=True)
        pkv = step.past_key_values; logits = step.logits[0,-1,:]
    return gen_tokenizer.decode(gen)

In [ ]:
import csv
rows=[]; total=0.0; t0=time.time()
for c in cases:
    out = generate_constrained(c['prompt'], c['forbidden'], int(c['max_new_tokens']))
    r = score_case(c['prompt'], c['forbidden'], int(c['max_new_tokens']), out)
    total += r['score']
    rows.append({'id':c['id'], 'output':out.replace('\n',' ').replace('\r',' '),
                 'fluency':r['fluency'], 'tokens':r['tokens']})
    print(f"[{c['id']:>2}] score={r['score']:.4f} tok={r['tokens']:3d}  {out[:60]!r}")
print(f'\nMean score (×100): {total/len(cases)*100:.4f}   ({time.time()-t0:.1f}s)')
with open('submission.csv','w',newline='',encoding='utf-8') as f:
    w=csv.DictWriter(f, fieldnames=['id','output','fluency','tokens']); w.writeheader(); w.writerows(rows)
print('wrote submission.csv', len(rows))

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)